# 📘 Домашнє завдання №17. Основи Explainable AI

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW17

### 📦 Дані
Використати датасет Titanic:
- target: `Survived`
- задача: **бінарна класифікація**

---

## 🔧 Частина 1. Підготовка даних

### 1.1. Препроцесинг
- обробити пропуски  
- закодувати категоріальні змінні  
- обрати релевантні фічі  

---

### 1.2. Feature Engineering (мінімум 2 нові фічі)

Приклади:
- `FamilySize = SibSp + Parch`
- `IsAlone`
- `Title` (з імені)

---

## 🌳 Частина 2. Explainable Boosting Machine (EBM)

### 2.1. Навчання моделі
- використати EBM (InterpretML)

### 2.2. Аналіз глобальних пояснень
- побудувати графіки впливу фіч (shape functions)

### 2.3. Інтерпретація
Для 3–5 фіч:
- як вони впливають на виживання?
- чи є нелінійності?

📌 **Питання:**
- чи є порогові ефекти (наприклад, вік)?
- які фічі найбільш важливі?

---

## 📜 Частина 3. RuleFit

### 3.1. Навчання RuleFit
- підготувати дані (one-hot encoding)
- витягнути правила

### 3.2. Аналіз правил
- вивести топ-10 правил
- пояснити їх

📌 **Питання:**
- які комбінації факторів ведуть до виживання?
- чи є правила, які легко інтерпретувати “людською мовою”?

---

## 📊 Частина 4. SHAP

### 4.1. Модель
- використати будь-яку (наприклад CatBoost / Random Forest)

### 4.2. SHAP аналіз
Побудувати:
- summary plot  
- bar plot  
- dependence plot (мінімум 2 фічі)  
- force або waterfall plot  

### 4.3. Інтерпретація

📌 Відповісти:
- які фічі найважливіші?
- як вони впливають?
- чи є взаємодії?


In [1]:
# # Silent installation or update
#
# # Clean cache
# !python3 -m pip cache purge -q
#
# # Force updating
# package_update = [
#     "pip",
#     "scikit-learn",
#     "pandas",
# ]
#
# for package_name in package_update:
#     !bash -c "python3 -m pip install -U '{package_name}' -q"
#
# # Install missing packages
# package_array = [
#     "jinja2",
#     "ipywidgets",
#     "nbformat",
#     "kagglehub[pandas-datasets]",
#     "numpy",
#     "matplotlib",
#     "scipy",
#     "statsmodels",
#     "plotly",
#     "joblib",
#     "nameparser",
# ]
#
# for package_name in package_array:
#     !bash -c "python3 -m pip show '{package_name}' > /dev/null 2>&1 || python3 -m pip install -U '{package_name}' -q"


In [2]:
# # Synchronization with remote source
#
# import shutil
# from pathlib import Path
#
# # Input data
hm_version = 17
#
# # Solution
# git_project_url = f"https://github.com/BogdanPinchuk/DataScience-PBY_HW{hm_version}.git"
# main_file_name = f"Bohdan_Pinchuk_DS_HW{hm_version}.ipynb"
#
# # upload all files
# current_path = !pwd
# current_path = current_path[0]
# parent_path = !dirname "$current_path"
# parent_path = parent_path[0]
# temp_path = f"{parent_path}/temp"
#
# # Clone data
# !rm -rf "$temp_path"
# !git clone "$git_project_url" "$temp_path"
#
# source = Path(temp_path)
# destination = Path(current_path)
# exclude = {main_file_name, ".git", ".idea"}
#
# for item in source.iterdir():
#     if item.name in exclude:
#         continue
#
#     target = destination / item.name
#     if item.is_dir():
#         shutil.copytree(item, target, dirs_exist_ok=True)
#     else:
#         shutil.copy2(item, target)
#
# # Clean temp folder
# !rm -rf "$temp_path"

## ✳️ Підготовка датасетів

In [3]:
# Downloading data

import apps.main as mn

# Input data
update_db = False
db_file_name = f"resources/store_hw{hm_version}.db"

# Solution
titanic_dataset = mn.download_and_extract_from_kagglehub("yasserh/titanic-dataset", "Titanic-Dataset.csv",
                                                         db_file_name, update_db)

# Print result
# display(titanic_dataset)


## ✅ Рішення 1

- обробити пропуски
- закодувати категоріальні змінні
- обрати релевантні фічі
- додати нові фічі

Приклади:
- `FamilySize = SibSp + Parch`
- `IsAlone`
- `Title` (з імені)

In [4]:
# Data analysis

import numpy as np
import pandas as pd
import apps.reporter as rpt
from IPython.display import display
from IPython.core.display import Markdown

# Input data
target_value = "Survived"
data_set = titanic_dataset
n_columns = data_set.columns.size
n_rows = data_set.index.size

# Solution
columns_df = pd.DataFrame(data_set.columns, columns=["Columns"])
types_df = pd.DataFrame(data_set.dtypes, columns=["Types"])

col_num_type = data_set.select_dtypes(include=np.number).columns
col_cat_type = data_set.select_dtypes(exclude=np.number).columns

empty_val_by_col = data_set.isnull().sum().to_frame(name='count')
is_empty_val = empty_val_by_col.values.sum() > 0

rp = rpt.Reporter()
rp.add_item("Кількість рядків", str(n_rows))
rp.add_item("Кількість об’єктів у датасеті", str(n_rows))
rp.add_item("Кількість стовпців", str(n_columns))
rp.add_item("Кількість числових ознак", str(col_num_type.size))
rp.add_item("Кількість категоріальних ознак", str(col_cat_type.size))
rp.add_item("Пропущені значення", 'є' if is_empty_val > 0 else 'немає')

# Print results
display(Markdown(f"### Аналіз даних"))
rp.print_pd_report("Параметри таблиці до обробки")

if is_empty_val > 0:
    display(empty_val_by_col.style.set_caption("Кількість пропусків"))
display(types_df.style.set_caption("Типи даних"))

display(data_set)

### Аналіз даних

Attribute,Result
Кількість рядків,891
Кількість об’єктів у датасеті,891
Кількість стовпців,12
Кількість числових ознак,7
Кількість категоріальних ознак,5
Пропущені значення,є


,count
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


,Types
PassengerId,int64
Survived,int64
Pclass,int64
Name,str
Sex,str
Age,float64
SibSp,int64
Parch,int64
Ticket,str
Fare,float64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [5]:
# Data preparation

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Input data
age_col_name = "Age"
fare_col_name = "Fare"
fil_num_col_list = ["Age"]
fil_cat_col_list = ["Embarked"]
drop_col_list = ["PassengerId", "Ticket", "Cabin"]

# Solution
titanic_dataset.drop(columns=drop_col_list, inplace=True)

preprocessor = ColumnTransformer([
    ("fil_num", SimpleImputer(strategy="median"), fil_num_col_list),
    ("fil_cat", SimpleImputer(strategy="most_frequent"), fil_cat_col_list),
], remainder="passthrough", verbose_feature_names_out=False)

preprocessor.set_output(transform="pandas")
transformed_data = preprocessor.fit_transform(titanic_dataset)
titanic_dataset = pd.DataFrame(transformed_data, columns=preprocessor.get_feature_names_out())

# Корегування даних
titanic_dataset[age_col_name] = titanic_dataset[age_col_name].round().astype(int)
titanic_dataset[fare_col_name] = titanic_dataset[fare_col_name].round(2).astype(float)

# Змінюємо тип
col_cat_type = titanic_dataset.select_dtypes(exclude=np.number).columns.tolist()
for col_name in col_cat_type:
    titanic_dataset[col_name] = titanic_dataset[col_name].astype('category')

data_set = titanic_dataset
n_columns = data_set.columns.size
n_rows = data_set.index.size

columns_df = pd.DataFrame(data_set.columns, columns=["Columns"])
types_df = pd.DataFrame(data_set.dtypes, columns=["Types"])

col_num_type = data_set.select_dtypes(include=np.number).columns
col_cat_type = data_set.select_dtypes(exclude=np.number).columns

empty_val_by_col = data_set.isnull().sum().to_frame(name='count')
is_empty_val = empty_val_by_col.values.sum() > 0

rp = rpt.Reporter()
rp.add_item("Кількість рядків", str(n_rows))
rp.add_item("Кількість об’єктів у датасеті", str(n_rows))
rp.add_item("Кількість стовпців", str(n_columns))
rp.add_item("Кількість числових ознак", str(col_num_type.size))
rp.add_item("Кількість категоріальних ознак", str(col_cat_type.size))
rp.add_item("Пропущені значення", 'є' if is_empty_val > 0 else 'немає')

# Print results
display(Markdown(f"### Підготовка даних (titanic_dataset)"))
rp.print_pd_report("Параметри таблиці без кодування")

if is_empty_val > 0:
    display(empty_val_by_col.style.set_caption("Кількість пропусків"))
display(types_df.style.set_caption("Типи даних"))
display(data_set)


### Підготовка даних (titanic_dataset)

Attribute,Result
Кількість рядків,891
Кількість об’єктів у датасеті,891
Кількість стовпців,9
Кількість числових ознак,6
Кількість категоріальних ознак,3
Пропущені значення,немає


,Types
Age,int64
Embarked,category
Survived,int64
Pclass,int64
Name,category
Sex,category
SibSp,int64
Parch,int64
Fare,float64


,Age,Embarked,Survived,Pclass,Name,Sex,SibSp,Parch,Fare
0,22,S,0,3,"Braund, Mr. Owen Harris",male,1,0,7.25
1,38,C,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,1,0,71.28
2,26,S,1,3,"Heikkinen, Miss. Laina",female,0,0,7.92
3,35,S,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,1,0,53.10
4,35,S,0,3,"Allen, Mr. William Henry",male,0,0,8.05
...,...,...,...,...,...,...,...,...,...
886,27,S,0,2,"Montvila, Rev. Juozas",male,0,0,13.00
887,19,S,1,1,"Graham, Miss. Margaret Edith",female,0,0,30.00
888,28,S,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,1,2,23.45
889,26,C,1,1,"Behr, Mr. Karl Howell",male,0,0,30.00


In [6]:
# Data preparation

import apps.main as mn
from nameparser import HumanName

# Input data
sex_col_name = "Sex"
sibsp_col_name = "SibSp"
parch_col_name = "Parch"
name_col_name = "Name"
family_col_name = "FamilySize"
alone_col_name = "IsAlone"
person_type_col_name = "PersonType"
title_col_name = "Title"
drop_col_list = [sibsp_col_name, parch_col_name, name_col_name]

# Solution
titanic_dataset[family_col_name] = (
    titanic_dataset.apply(lambda df_row: df_row[sibsp_col_name] + df_row[parch_col_name] + 1,
                          axis=1).astype(int))
titanic_dataset[alone_col_name] = (titanic_dataset[family_col_name] == 1).astype(int)
titanic_dataset[person_type_col_name] = titanic_dataset.apply(
    lambda df_row: mn.get_person_types(df_row[sex_col_name], df_row[age_col_name]), axis=1).astype('category')

titanic_dataset[name_col_name] = titanic_dataset[name_col_name].str.replace(r'[\(\)]', '', regex=True)
titanic_dataset[title_col_name] = titanic_dataset.apply(lambda df_row: HumanName(df_row[name_col_name]).first.strip(),
                                                        axis=1).astype('category')

titanic_dataset.drop(columns=drop_col_list, inplace=True)

data_set = titanic_dataset
n_columns = data_set.columns.size
n_rows = data_set.index.size

columns_df = pd.DataFrame(data_set.columns, columns=["Columns"])
types_df = pd.DataFrame(data_set.dtypes, columns=["Types"])

col_num_type = data_set.select_dtypes(include=np.number).columns
col_cat_type = data_set.select_dtypes(exclude=np.number).columns

cat_type_df = data_set[col_cat_type]

empty_val_by_col = data_set.isnull().sum().to_frame(name='count')
is_empty_val = empty_val_by_col.values.sum() > 0

rp = rpt.Reporter()
rp.add_item("Кількість рядків", str(n_rows))
rp.add_item("Кількість об’єктів у датасеті", str(n_rows))
rp.add_item("Кількість стовпців", str(n_columns))
rp.add_item("Кількість числових ознак", str(col_num_type.size))
rp.add_item("Кількість категоріальних ознак", str(col_cat_type.size))
rp.add_item("Пропущені значення", 'є' if is_empty_val > 0 else 'немає')

# Print results
display(Markdown(f"### Підготовка даних (titanic_dataset)"))
rp.print_pd_report("Параметри таблиці без кодування")

if is_empty_val > 0:
    display(empty_val_by_col.style.set_caption("Кількість пропусків"))
display(types_df.style.set_caption("Типи даних"))

display(cat_type_df.describe().T.style.set_caption("Статистика для категоріальних змінних"))

for col_name in col_cat_type:
    display(pd.concat([
        cat_type_df[col_name].value_counts(),
        cat_type_df[col_name].value_counts(normalize=True)
    ], axis=1).head(10).style.set_caption(f"Категорія: \"{col_name}\""))

display(data_set)


### Підготовка даних (titanic_dataset)

Attribute,Result
Кількість рядків,891
Кількість об’єктів у датасеті,891
Кількість стовпців,10
Кількість числових ознак,6
Кількість категоріальних ознак,4
Пропущені значення,немає


,Types
Age,int64
Embarked,category
Survived,int64
Pclass,int64
Sex,category
Fare,float64
FamilySize,int64
IsAlone,int64
PersonType,category
Title,category


,count,unique,top,freq
Embarked,891,3,S,646
Sex,891,2,male,577
PersonType,891,5,man,519
Title,891,436,William,48


,count,proportion
Embarked,,
S,646,0.725028
C,168,0.188552
Q,77,0.086420


,count,proportion
Sex,,
male,577,0.647587
female,314,0.352413


,count,proportion
PersonType,,
man,519,0.582492
woman,259,0.290685
girl,45,0.050505
boy,44,0.049383
baby,24,0.026936


,count,proportion
Title,,
William,48,0.053872
John,30,0.033670
Thomas,19,0.021324
Charles,16,0.017957
George,16,0.017957
Henry,15,0.016835
James,15,0.016835
Edward,13,0.014590
Frederick,13,0.014590


,Age,Embarked,Survived,Pclass,Sex,Fare,FamilySize,IsAlone,PersonType,Title
0,22,S,0,3,male,7.25,2,0,man,Owen
1,38,C,1,1,female,71.28,2,0,woman,John
2,26,S,1,3,female,7.92,1,1,woman,Laina
3,35,S,1,1,female,53.10,2,0,woman,Jacques
4,35,S,0,3,male,8.05,1,1,man,William
...,...,...,...,...,...,...,...,...,...,...
886,27,S,0,2,male,13.00,1,1,man,Juozas
887,19,S,1,1,female,30.00,1,1,woman,Margaret
888,28,S,0,3,female,23.45,4,0,woman,Catherine
889,26,C,1,1,male,30.00,1,1,man,Karl


In [8]:
# Data preparation

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# Input data
fil_cat_col_list = ["Embarked", "Sex", "IsAlone", "PersonType"]
fil_lab_col_list = ["Title"]

# Solution
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype="int64"), fil_cat_col_list),
    ("lab", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype="int64"), fil_lab_col_list),
], remainder="passthrough", verbose_feature_names_out=False)

preprocessor.set_output(transform="pandas")
transformed_data = preprocessor.fit_transform(titanic_dataset)
titanic_dataset_ohe = pd.DataFrame(transformed_data, columns=preprocessor.get_feature_names_out())

data_set = titanic_dataset_ohe
n_columns = data_set.columns.size
n_rows = data_set.index.size

columns_df = pd.DataFrame(data_set.columns, columns=["Columns"])
types_df = pd.DataFrame(data_set.dtypes, columns=["Types"])

col_num_type = data_set.select_dtypes(include=np.number).columns
col_cat_type = data_set.select_dtypes(exclude=np.number).columns

cat_type_df = data_set[col_cat_type]

empty_val_by_col = data_set.isnull().sum().to_frame(name='count')
is_empty_val = empty_val_by_col.values.sum() > 0

rp = rpt.Reporter()
rp.add_item("Кількість рядків", str(n_rows))
rp.add_item("Кількість об’єктів у датасеті", str(n_rows))
rp.add_item("Кількість стовпців", str(n_columns))
rp.add_item("Кількість числових ознак", str(col_num_type.size))
rp.add_item("Кількість категоріальних ознак", str(col_cat_type.size))
rp.add_item("Пропущені значення", 'є' if is_empty_val > 0 else 'немає')

# Print results
display(Markdown(f"### Підготовка даних (titanic_dataset_ohe)"))
rp.print_pd_report("Параметри таблиці з кодуванням")

if is_empty_val > 0:
    display(empty_val_by_col.style.set_caption("Кількість пропусків"))
display(types_df.style.set_caption("Типи даних"))
display(data_set)

### Підготовка даних (titanic_dataset_ohe)

Attribute,Result
Кількість рядків,891
Кількість об’єктів у датасеті,891
Кількість стовпців,18
Кількість числових ознак,18
Кількість категоріальних ознак,0
Пропущені значення,немає


,Types
Embarked_C,int64
Embarked_Q,int64
Embarked_S,int64
Sex_female,int64
Sex_male,int64
IsAlone_0,int64
IsAlone_1,int64
PersonType_baby,int64
PersonType_boy,int64
PersonType_girl,int64


,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male,IsAlone_0,IsAlone_1,PersonType_baby,PersonType_boy,PersonType_girl,PersonType_man,PersonType_woman,Title,Age,Survived,Pclass,Fare,FamilySize
0,0,0,1,0,1,1,0,0,0,0,1,0,337,22,0,3,7.25,2
1,1,0,0,1,0,1,0,0,0,0,0,1,217,38,1,1,71.28,2
2,0,0,1,1,0,0,1,0,0,0,0,1,246,26,1,3,7.92,1
3,0,0,1,1,0,1,0,0,0,0,0,1,202,35,1,1,53.10,2
4,0,0,1,0,1,0,1,0,0,0,1,0,431,35,0,3,8.05,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,0,1,0,1,0,1,0,0,0,1,0,230,27,0,2,13.00,1
887,0,0,1,1,0,0,1,0,0,0,0,1,282,19,1,1,30.00,1
888,0,0,1,1,0,1,0,0,0,0,0,1,64,28,0,3,23.45,4
889,1,0,0,0,1,0,1,0,0,0,1,0,232,26,1,1,30.00,1
